In [ ]:
import os
import random
from datetime import datetime, timedelta
from faker import Faker
import oracledb

# Inicializar Faker en español
fake = Faker('es_ES')

# Configuración de conexión desde variables de entorno
ORACLE_HOST = os.getenv('ORACLE_HOST')
ORACLE_PORT = os.getenv('ORACLE_PORT')
ORACLE_SERVICE = os.getenv('ORACLE_SERVICE')
ORACLE_USER = os.getenv('ORACLE_USER')
ORACLE_PASSWORD = os.getenv('ORACLE_PASSWORD')

# Configuración de volúmenes
NUM_CATEGORIES = 8
NUM_PRODUCTS = 75
NUM_CUSTOMERS = 500
NUM_ORDERS = 2000
REVIEW_RATIO = 0.35  # 35% de los ítems entregados reciben reseña

def get_connection():
    dsn = f"{ORACLE_HOST}:{ORACLE_PORT}/{ORACLE_SERVICE}"
    return oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=dsn)

def main():
    conn = get_connection()
    cursor = conn.cursor()
    print("Conexión establecida con Oracle DB.")

    try:
        # ------------------------------------------------------------------
        # 1. CATEGORÍAS
        # ------------------------------------------------------------------
        print("Generando Categorías...")
        categories_data = [
            (1, 'Electrónica', 'Dispositivos electrónicos, gadgets y accesorios'),
            (2, 'Informática', 'Ordenadores, componentes y periféricos'),
            (3, 'Hogar y Cocina', 'Muebles, decoración y electrodomésticos'),
            (4, 'Ropa y Moda', 'Ropa para hombre, mujer y niños'),
            (5, 'Deportes y Exterior', 'Material deportivo y ropa de montaña'),
            (6, 'Libros y Papelería', 'Libros físicos, e-books y material de oficina'),
            (7, 'Juegos y Juguetes', 'Juegos de mesa, consolas y juguetes'),
            (8, 'Belleza y Cuidado Personal', 'Cosmética, perfumes y salud')
        ]
        cursor.executemany(
            "INSERT INTO CATEGORIES (CATEGORY_ID, NAME, DESCRIPTION) VALUES (:1, :2, :3)",
            categories_data
        )

        # ------------------------------------------------------------------
        # 2. PRODUCTOS
        # ------------------------------------------------------------------
        print("Generando Productos...")
        products = []
        products_dict = {}  # Para guardar PVP y COST para los pedidos

        for p_id in range(1, NUM_PRODUCTS + 1):
            cat_id = random.randint(1, NUM_CATEGORIES)
            name = fake.catch_phrase()[:100]
            desc = fake.text(max_nb_chars=250)
            cost = round(random.uniform(5.0, 300.0), 2)
            # Margen de beneficio entre 20% y 80%
            pvp = round(cost * random.uniform(1.2, 1.8), 2)
            stock = random.randint(0, 500)
            status = 1 if random.random() > 0.05 else 0  # 95% activos

            products.append((p_id, cat_id, name, desc, pvp, cost, stock, status))
            products_dict[p_id] = {'pvp': pvp, 'cost': cost}

        cursor.executemany(
            """INSERT INTO PRODUCTS 
               (PRODUCT_ID, CATEGORY_ID, NAME, DESCRIPTION, PVP, COST, STOCK, STATUS) 
               VALUES (:1, :2, :3, :4, :5, :6, :7, :8)""",
            products
        )

        # ------------------------------------------------------------------
        # 3. CLIENTES
        # ------------------------------------------------------------------
        print("Generando Clientes...")
        customers = []
        customer_reg_dates = {}
        canales = ['WEB', 'APP_IOS', 'APP_ANDROID', 'TIENDA_FISICA', 'PUBLICIDAD']

        start_date = datetime.now() - timedelta(days=730)  # Últimos 2 años

        for c_id in range(1, NUM_CUSTOMERS + 1):
            first_name = fake.first_name()
            last_name = fake.last_name()
            email = f"{first_name.lower()}.{last_name.lower()}{c_id}@example.com"
            country = 'España'
            city = fake.city()
            canal = random.choice(canales)
            reg_date = fake.date_time_between(start_date=start_date, end_date='now')
            
            customers.append((c_id, first_name, last_name, email, country, city, canal, reg_date))
            customer_reg_dates[c_id] = reg_date

        cursor.executemany(
            """INSERT INTO CUSTOMERS 
               (CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, COUNTRY, CITY, CANAL, REGISTRATION_DATE) 
               VALUES (:1, :2, :3, :4, :5, :6, :7, :8)""",
            customers
        )

        # ------------------------------------------------------------------
        # 4. PEDIDOS, LÍNEAS DE PEDIDO Y PAGOS
        # ------------------------------------------------------------------
        print("Generando Pedidos, Líneas de Pedido y Pagos...")
        orders = []
        order_items = []
        payments = []
        
        # Guardaremos referencias de ítems entregados para generar reseñas luego
        delivered_items_for_review = []
        
        payment_methods = ['TARJETA', 'PAYPAL', 'TRANSFERENCIA', 'BIZUM']
        statuses = ['ENTREGADO', 'ENVIADO', 'PENDIENTE', 'CANCELADO']
        status_weights = [0.70, 0.15, 0.10, 0.05]

        product_ids_list = list(products_dict.keys())
        payment_id_counter = 1

        for o_id in range(1, NUM_ORDERS + 1):
            c_id = random.randint(1, NUM_CUSTOMERS)
            c_reg_date = customer_reg_dates[c_id]
            
            # Fecha de pedido posterior al registro del cliente
            order_date = fake.date_time_between(start_date=c_reg_date, end_date='now')
            
            status = random.choices(statuses, weights=status_weights)[0]
            
            # Gestión coherente de fechas según estado
            send_date = None
            receive_date = None
            if status in ['ENVIADO', 'ENTREGADO']:
                send_date = order_date + timedelta(days=random.randint(1, 3))
            if status == 'ENTREGADO':
                receive_date = send_date + timedelta(days=random.randint(1, 5))

            send_city = fake.city()
            send_country = 'España'
            send_postal = fake.postcode()
            send_address = fake.street_address()

            # Generar entre 1 y 4 ítems distintos por pedido (Media ~2.5 -> ~4500 líneas en total)
            num_items = random.randint(1, 4)
            selected_products = random.sample(product_ids_list, k=num_items)
            
            total_amount = 0.0

            for item_seq, p_id in enumerate(selected_products, start=1):
                quantity = random.randint(1, 5)
                unit_price = products_dict[p_id]['pvp']
                # Descuento entre 0 y 15%
                discount = round((unit_price * quantity) * (random.choice([0, 0, 0, 0.05, 0.10, 0.15])), 2)
                
                line_total = (unit_price * quantity) - discount
                total_amount += line_total

                order_items.append((o_id, item_seq, p_id, quantity, unit_price, discount))

                if status == 'ENTREGADO':
                    delivered_items_for_review.append({
                        'order_id': o_id,
                        'order_items_id': item_seq,
                        'product_id': p_id,
                        'customer_id': c_id,
                        'receive_date': receive_date
                    })

            total_amount = round(total_amount, 2)

            orders.append((
                o_id, c_id, order_date, send_city, send_country, 
                send_postal, send_address, send_date, receive_date, 
                total_amount, status
            ))

            # Generar Pago coherente
            if status != 'CANCELADO':
                pay_status = 'COMPLETADO' if status in ['ENTREGADO', 'ENVIADO'] else 'PENDIENTE'
                pay_date = order_date + timedelta(minutes=random.randint(1, 60))
                pay_method = random.choice(payment_methods)
                
                payments.append((
                    payment_id_counter, o_id, pay_date, total_amount, pay_method, pay_status
                ))
                payment_id_counter += 1

        cursor.executemany(
            """INSERT INTO ORDERS 
               (ORDER_ID, CUSTOMER_ID, ORDER_DATE, SEND_CITY, SEND_COUNTRY, 
                SEND_POSTAL_CODE, SEND_ADDRESS, SEND_DATE, RECEIVE_DATE, TOTAL_AMOUNT, STATUS) 
               VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9, :10, :11)""",
            orders
        )

        cursor.executemany(
            """INSERT INTO ORDER_ITEMS 
               (ORDER_ID, ORDER_ITEMS_ID, PRODUCT_ID, QUANTITY, UNIT_PRICE, DISCOUNT) 
               VALUES (:1, :2, :3, :4, :5, :6)""",
            order_items
        )

        cursor.executemany(
            """INSERT INTO PAYMENTS 
               (PAYMENT_ID, ORDER_ID, PAYMENT_DATE, AMOUNT, PAYMENT_METHOD, STATUS) 
               VALUES (:1, :2, :3, :4, :5, :6)""",
            payments
        )

        # ------------------------------------------------------------------
        # 5. RESEÑAS (REVIEWS)
        # ------------------------------------------------------------------
        print("Generando Reseñas...")
        reviews = []
        num_reviews = int(len(delivered_items_for_review) * REVIEW_RATIO)
        selected_for_review = random.sample(delivered_items_for_review, k=num_reviews)

        for r_id, item in enumerate(selected_for_review, start=1):
            rating = random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.05, 0.15, 0.35, 0.40])[0]
            comment = fake.paragraph(nb_sentences=2) if random.random() > 0.2 else None
            # Fecha de reseña posterior a la recepción del pedido
            review_date = item['receive_date'] + timedelta(days=random.randint(1, 30))

            reviews.append((
                r_id, item['product_id'], item['customer_id'], 
                item['order_id'], item['order_items_id'], 
                rating, comment, review_date
            ))

        cursor.executemany(
            """INSERT INTO REVIEWS 
               (REVIEW_ID, PRODUCT_ID, CUSTOMER_ID, ORDER_ID, ORDER_ITEMS_ID, RATING, COMMENT_TEXT, REVIEW_DATE) 
               VALUES (:1, :2, :3, :4, :5, :6, :7, :8)""",
            reviews
        )

        # Confirmar cambios
        conn.commit()
        print(f"\n¡Carga completada exitosamente!")
        print(f" - Categorías: {len(categories_data)}")
        print(f" - Productos: {len(products)}")
        print(f" - Clientes: {len(customers)}")
        print(f" - Pedidos: {len(orders)}")
        print(f" - Líneas de pedido: {len(order_items)}")
        print(f" - Pagos: {len(payments)}")
        print(f" - Reseñas: {len(reviews)}")

    except Exception as e:
        conn.rollback()
        print(f"\n[ERROR] Ocurrió un fallo durante la inserción. Transacción revertida.")
        print(f"Detalle: {e}")
    finally:
        cursor.close()
        conn.close()

if __name__ == '__main__':
    main()

Conexión establecida con Oracle DB.
Generando Categorías...
Generando Productos...
Generando Clientes...
Generando Pedidos, Líneas de Pedido y Pagos...
Generando Reseñas...

¡Carga completada exitosamente!
 - Categorías: 8
 - Productos: 75
 - Clientes: 500
 - Pedidos: 2000
 - Líneas de pedido: 5016
 - Pagos: 1923
 - Reseñas: 1215
